In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Sample user-item ratings
data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4],
    'movie': ['Batman', 'Harry Potter', 'Shrek', 'Batman', 'Memento', 'Shrek', 'Memento', 'Harry Potter'],
    'rating': [5, 4, 2, 5, 3, 4, 5, 4]
}
df = pd.DataFrame(data)


In [3]:
user_movie_matrix = df.pivot_table(index='user_id', columns='movie', values='rating').fillna(0)


In [4]:
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)


In [5]:
def recommend_movies(user_id, user_movie_matrix, user_similarity_df, n_recommendations=2):
    # Get the user's ratings
    user_ratings = user_movie_matrix.loc[user_id]
    # Find movies the user hasn't rated
    unrated_movies = user_ratings[user_ratings == 0].index
    # Calculate weighted average score for each unrated movie
    scores = {}
    for movie in unrated_movies:
        # Ratings of this movie by OTHER users (exclude current user)
        movie_ratings = user_movie_matrix[movie].drop(user_id)  # Fix here
        # Similarity scores (exclude current user)
        similarities = user_similarity_df.loc[user_id].drop(user_id)
        # Weighted sum
        weighted_sum = np.dot(movie_ratings, similarities)
        sim_sum = np.sum(similarities)
        if sim_sum > 0:
            scores[movie] = weighted_sum / sim_sum
    # Sort and return top recommendations
    recommended = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    return [movie for movie, score in recommended]


In [6]:
for user in user_movie_matrix.index:
    recs = recommend_movies(user, user_movie_matrix, user_similarity_df, n_recommendations=2)
    print(f"Recommendations for User {user}: {recs}")


Recommendations for User 1: ['Memento']
Recommendations for User 2: ['Shrek', 'Harry Potter']
Recommendations for User 3: ['Batman', 'Harry Potter']
Recommendations for User 4: ['Batman', 'Shrek']
